In [1]:
%cd ..
from pathlib import Path
import csv
from collections import Counter
import plotly.graph_objects as go
import numpy as np
from sklearn.metrics import confusion_matrix
from Recognition.live_recognition import run_live_recognition

model_name = "TransformerEncoder"
model_checkpoint_path = Path(f"Models/{model_name}_Checkpoints/best_encoder.pt")
faiss_index_path = f"Recognition/{model_name}_db_custom/index.faiss"
faiss_labels_path = f"Recognition/{model_name}_db_custom/index_labels.npz"

c:\Users\mjane\Documents\GitHub\LSTM-FAISS-DTW


c:\Users\mjane\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [ ]:
results = run_live_recognition(
    is_diagnostic= True, 
    is_custom_database= True,
    labels= ["WORK", "SLEEP", "ME", "HOME", "READ"],               #labels=None (all), labels=["BOOK", "READ"] (selected)
    repeats= 3,
    shuffle= True,
    camera_index= 0,
    top_k= 1,
    checkpoint_path = model_checkpoint_path
)

Model works on: cpu
FAISS index: 10 vectors, 10 gesture classes
[1/15] true=HOME pred=HOME correct=True
[2/15] true=HOME pred=DRINK correct=False
[3/15] true=READ pred=READ correct=True
[4/15] true=SLEEP pred=SLEEP correct=True
[5/15] true=HOME pred=HOME correct=True
[6/15] true=READ pred=DRINK correct=False
[7/15] true=ME pred=ME correct=True
[8/15] true=ME pred=ME correct=True
[9/15] true=WORK pred=WORK correct=True
[10/15] true=ME pred=DRINK correct=False
[11/15] true=WORK pred=WORK correct=True
[12/15] true=READ pred=READ correct=True
[13/15] true=SLEEP pred=SLEEP correct=True
[14/15] true=SLEEP pred=SLEEP correct=True
[15/15] true=WORK pred=WORK correct=True
Accuracy: 80.00% (12/15)


In [3]:
y_true = results["true_label"]
y_pred = results["predicted_label"]

labels_names = sorted(set(y_true) | set(y_pred))

cm = confusion_matrix(y_true, y_pred, labels=labels_names)
row_sums = cm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)

fig = go.Figure(data=go.Heatmap(
    z = cm_norm,
    zmin = 0,
    zmax = 1,
    x = labels_names,
    y = labels_names,
    colorscale="turbo",
    text=cm,
    texttemplate="%{text}",
    hovertemplate="True: %{y}<br>Pred: %{x}<extra></extra>"
))

fig.update_layout(
    title="Prediction in Best Epoch",
    xaxis_title="Predicted label",
    yaxis_title="True label",
    width=1200,
    height=1000,
    margin=dict(l=150, r=50, t=80, b=150)
)

fig.show()